# Estimating the Standard Deviation of a Prediction Using the Bootstrap

**ISLP Chapter 5, Conceptual Exercise 4**

Suppose that we use some statistical learning method to make a prediction for the response $Y$ for a particular value of the predictor $X$. We describe how to estimate the standard deviation of our prediction using the bootstrap.

## Answer

Assuming our dataset consists of independent observations (as opposed to something like time series data, in which observations that are close together in time are generally not independent and using a strategy such as a block bootstrap would be preferable), we can use the **bootstrap** to estimate the standard deviation of our prediction.

Suppose our original data set had $n$ observations. To perform the bootstrap for this situation, we randomly sample, **with replacement**, $n$ observations from our original set of observations to use as a training set for our statistical learning method and obtain a bootstrap model $\mathcal{M}^{*i}$. We then use the model $\mathcal{M}^{*i}$ to predict the response $\hat{Y}^{*i}$ for the value of the predictor $X$. We repeat this procedure $B$ times for some large value of $B$ in order to produce $B$ different bootstrap models $\mathcal{M}^{*1}, \ldots, \mathcal{M}^{*B}$ and $B$ corresponding $Y$ estimates, $\hat{Y}^{*1}, \ldots, \hat{Y}^{*B}$. Once we have done this, we can use the formula for the [sample standard deviation](https://en.wikipedia.org/wiki/Standard_deviation#Corrected_sample_standard_deviation) to compute an estimate of the standard deviation of our prediction.

$$\text{SE}_B(\hat{Y}) = \sqrt{\frac{1}{B-1} \sum_{j=1}^{B} \left( \hat{Y}^{*j} - \frac{1}{B} \sum_{k=1}^{B} \hat{Y}^{*k} \right)^2 }$$

## Demonstration

Below we illustrate this procedure using a simple example with polynomial regression on synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Generate synthetic data
np.random.seed(42)
n = 100
X = np.random.uniform(0, 10, n)
Y = 2 + 3 * X - 0.5 * X**2 + np.random.normal(0, 3, n)

# The particular value of X at which we want to predict Y
X_new = np.array([5.0])

In [ ]:
# Bootstrap procedure
B = 1000  # number of bootstrap replicates
predictions = np.empty(B)

rng = np.random.default_rng(0)

for b in range(B):
    # Step 1: Sample n observations with replacement
    idx = rng.choice(n, n, replace=True)
    X_boot = X[idx]
    Y_boot = Y[idx]
    
    # Step 2: Fit the statistical learning method (here, quadratic regression)
    poly = PolynomialFeatures(degree=2)
    X_boot_poly = poly.fit_transform(X_boot.reshape(-1, 1))
    model = LinearRegression().fit(X_boot_poly, Y_boot)
    
    # Step 3: Predict Y at the value X = 5.0
    X_new_poly = poly.transform(X_new.reshape(-1, 1))
    predictions[b] = model.predict(X_new_poly)[0]

# Step 4: Compute the bootstrap standard error (sample standard deviation)
se_boot = np.sqrt(1 / (B - 1) * np.sum((predictions - predictions.mean())**2))
# Equivalently: se_boot = predictions.std(ddof=1)

print(f"Bootstrap estimate of SD of prediction at X=5: {se_boot:.4f}")
print(f"Mean bootstrap prediction: {predictions.mean():.4f}")
print(f"Verify with np.std(ddof=1): {predictions.std(ddof=1):.4f}")

In [ ]:
# Visualize the bootstrap distribution of predictions
plt.figure(figsize=(8, 5))
plt.hist(predictions, bins=40, edgecolor='white', alpha=0.7, color='steelblue')
plt.axvline(predictions.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {predictions.mean():.2f}')
plt.xlabel(r'$\hat{Y}^*$ at $X = 5$', fontsize=13)
plt.ylabel('Frequency', fontsize=13)
plt.title(f'Bootstrap Distribution of Predictions (SE = {se_boot:.3f})', fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('bootstrap_prediction_hist.png', dpi=150)
plt.show()